# GlobeX Machine Learning Lab: Document Intelligence & Cross-Doc Reconciliation
## Trade Document Corpus EDA, Key-Value Field Extraction & SHA-256 Anchoring

**Author**: GlobeX Core ML Team  
**Dataset**: Global Trade Clearance Document Corpus (`03_document_intelligence_eda.csv`, `03_document_intelligence_dl.csv`)  
**Objective**: Extract structured parameters from Commercial Invoices, Bills of Lading, and Phytosanitary certificates, perform cross-document consistency reconciliation (detecting weight variances and date sequence anomalies), and anchor SHA-256 digital proofs.

---
### Workflow Outline:
1. **Environment Setup & Document Corpus Ingestion**
2. **Document Corpus Exploratory Data Analysis (EDA)**
   - Breakdown by document type (Invoice, BoL, Phytosanitary, Certificate of Origin)
   - Key field completeness & missingness analysis
   - Token count & OCR extraction confidence distributions
3. **Structured Field Extraction Pipeline**
   - Invoice #, Contract USD Value, Net Weight MT, HS Code, Shipper, Consignee
4. **Cross-Document Consistency Reconciler**
   - Weight discrepancy tolerance verification (tolerance $\le 0.5\%$)
   - Date sequence integrity check (Phyto $\le$ BoL $\le$ Customs Entry)
5. **Cryptographic SHA-256 Digital Fingerprinting**
6. **Verification Verdict & Export**


In [1]:
import os
import sys
import json
import hashlib
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print('Document intelligence environment ready.')


Document intelligence environment ready.


### Loading Trade Document Corpus


In [2]:
data_candidates = [
    '../backend/brain/brain_prev/data_pipeline/data/final_csv/03_document_intelligence_eda.csv',
    'backend/brain/brain_prev/data_pipeline/data/final_csv/03_document_intelligence_eda.csv',
    '../backend/brain/brain_prev/data_pipeline/data/final_csv/03_document_intelligence_dl.csv',
    'backend/brain/brain_prev/data_pipeline/data/final_csv/03_document_intelligence_dl.csv',
]

dataset_path = None
for p in data_candidates:
    if os.path.exists(p):
        dataset_path = p
        break

if dataset_path is None:
    dataset_path = 'backend/brain/brain_prev/data_pipeline/data/final_csv/05_rag_evidence.csv'

print(f'Loading document intelligence data from: {dataset_path}')
df = pd.read_csv(dataset_path)
print(f'Loaded {df.shape[0]:,} document records.')


Loading document intelligence data from: backend/brain/brain_prev/data_pipeline/data/final_csv/03_document_intelligence_eda.csv
Loaded 91 document records.


## 2. Document Corpus Exploratory Data Analysis (EDA)


In [3]:
display(df.head(5))
print('\n--- DOCUMENT CORPUS METADATA ---')
df.info()


,document_id,source_dataset,source_version,split,image_reference,language,document_type,token_index,token,x0,y0,x1,y1,entity_label,linked_token_ids,key,value
0,DOC_FUNSD_TRAIN_001,FUNSD,v1.0,train,data/raw/documents/funsd/images/train_DOC_FUNS...,en,CERTIFICATE_OF_ORIGIN,0,CERTIFICATE,120,45,310,75,HEADER,[],TITLE,CERTIFICATE
1,DOC_FUNSD_TRAIN_001,FUNSD,v1.0,train,data/raw/documents/funsd/images/train_DOC_FUNS...,en,CERTIFICATE_OF_ORIGIN,1,OF,320,45,360,75,HEADER,[],TITLE,OF
2,DOC_FUNSD_TRAIN_001,FUNSD,v1.0,train,data/raw/documents/funsd/images/train_DOC_FUNS...,en,CERTIFICATE_OF_ORIGIN,2,ORIGIN,370,45,490,75,HEADER,[],TITLE,ORIGIN
3,DOC_FUNSD_TRAIN_001,FUNSD,v1.0,train,data/raw/documents/funsd/images/train_DOC_FUNS...,en,CERTIFICATE_OF_ORIGIN,3,Exporter:,50,110,140,130,QUESTION,"[4, 5, 6, 7]",EXPORTER,NaN
4,DOC_FUNSD_TRAIN_001,FUNSD,v1.0,train,data/raw/documents/funsd/images/train_DOC_FUNS...,en,CERTIFICATE_OF_ORIGIN,4,Bharat,150,110,210,130,ANSWER,[],NaN,Bharat



--- DOCUMENT CORPUS METADATA ---
<class 'pandas.DataFrame'>
RangeIndex: 91 entries, 0 to 90
Data columns (total 17 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   document_id       91 non-null     str  
 1   source_dataset    91 non-null     str  
 2   source_version    91 non-null     str  
 3   split             91 non-null     str  
 4   image_reference   91 non-null     str  
 5   language          91 non-null     str  
 6   document_type     91 non-null     str  
 7   token_index       91 non-null     int64
 8   token             91 non-null     str  
 9   x0                91 non-null     int64
 10  y0                91 non-null     int64
 11  x1                91 non-null     int64
 12  y1                91 non-null     int64
 13  entity_label      91 non-null     str  
 14  linked_token_ids  91 non-null     str  
 15  key               59 non-null     str  
 16  value             58 non-null     str  
dtypes: int64(5), s

## 3. Cross-Document Reconciliation Matrix & Discrepancy Detection


In [4]:
# Document reconciliation test case simulation
docs = {
    'Commercial Invoice': {'doc_id': 'INV-2026-8891', 'hs_code': '1006.30', 'weight_mt': 500.0, 'val_usd': 550000, 'date': '2026-08-20'},
    'Bill of Lading': {'doc_id': 'BL-MAEU-98214', 'hs_code': '1006.30', 'weight_mt': 500.0, 'val_usd': 550000, 'date': '2026-08-22'},
    'Phytosanitary Certificate': {'doc_id': 'APEDA-PHY-2026-441', 'hs_code': '1006.30', 'weight_mt': 500.0, 'val_usd': 550000, 'date': '2026-08-19'}
}

doc_df = pd.DataFrame(docs).T
print('=== EXTRACTED TRADE DOCUMENT SET ===')
display(doc_df)

# Cross-Check Consistency
weights = doc_df['weight_mt'].values
hs_codes = doc_df['hs_code'].values

weight_match = (weights == weights[0]).all()
hs_match = (hs_codes == hs_codes[0]).all()

print(f'\nCross-Document Weight Reconciliation: {"PASSED (100% Match)" if weight_match else "FAILED"}')
print(f'Cross-Document HS Code Reconciliation: {"PASSED (100% Match)" if hs_match else "FAILED"}')


=== EXTRACTED TRADE DOCUMENT SET ===


,doc_id,hs_code,weight_mt,val_usd,date
Commercial Invoice,INV-2026-8891,1006.30,500.0,550000,2026-08-20
Bill of Lading,BL-MAEU-98214,1006.30,500.0,550000,2026-08-22
Phytosanitary Certificate,APEDA-PHY-2026-441,1006.30,500.0,550000,2026-08-19



Cross-Document Weight Reconciliation: PASSED (100% Match)
Cross-Document HS Code Reconciliation: PASSED (100% Match)


## 4. Cryptographic SHA-256 Digital Fingerprinting


In [5]:
fingerprints = {}
for name, data in docs.items():
    payload = json.dumps(data, sort_keys=True).encode('utf-8')
    sha = hashlib.sha256(payload).hexdigest()
    fingerprints[name] = sha
    print(f'{name} SHA-256: 0x{sha}')

print('\nAll documents cryptographically signed and ready for blockchain anchoring.')


Commercial Invoice SHA-256: 0x5cb07162e763d3d38797e90b7413c5830a388acc4c7e7395b9f080286da3d7fe
Bill of Lading SHA-256: 0xf91ede5ec173903d67175bdcceeb9243715f639fa084781780a8875f0396b3b4
Phytosanitary Certificate SHA-256: 0x766b3fe873ed76e6481b5b926a18c388f51428067818b83900aa900e0c2d432e

All documents cryptographically signed and ready for blockchain anchoring.
